In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from saddler_simple import erbspace_tch, get_gammatone_filter_coefs_tch, FIRGammatoneFilterbank, SigmoidRateLevelFunction, SpikeGeneratorBinomial_Tch, _hilbert_envelope
from z_ref_phaselocknet.util_cochlea import *
from z_ref_phaselocknet.util_signal import fir_gammatone_filterbank, get_gammatone_impulse_responses, erbspace, tf_hilbert
from utils.read_listen_save import read_stereo_audio
from utils.plotter import plot_2d
import json
import plotly.express as px

q# Basic functions

In [ ]:
def to_numpy_from_tf(tensor):
    return torch.tensor(tensor.numpy()).squeeze().numpy()

def to_numpy(tensor):
    return tensor.detach().squeeze().numpy()

def to_db(gm):
    gm_abs = np.abs(gm)
    gm_db = 20 * np.log10(gm_abs/20e-6)
    return gm_db

def plot_diff(t_bin, f_bin, tf, tch):
    tf_np = to_numpy_from_tf(tf)
    tch_np = to_numpy(tch)
    plot_2d(x=t_bin, y=f_bin, z=tf_np, name='tensorflow output', title=f'tensorflow cochela',
                        xaxis_title='Time (sec)', yaxis_title='Frequency (Hz)', width=800, height=600,)
    plot_2d(x=t_bin, y=f_bin, z=tch_np, name='pytorch output', title=f'pytorch output',
                        xaxis_title='Time (sec)', yaxis_title='Frequency (Hz)', width=800, height=600,)
    plot_2d(x=t_bin, y=f_bin, z=tf_np-tch_np, name='Difference output', title=f'Difference output',
                        xaxis_title='Time (sec)', yaxis_title='Frequency (Hz)', width=800, height=600,)

# 1. Check gammatone filters & output

### Data loading

In [ ]:
data_fpath = '../binaural_data/azim_0_elev_0_sample_0.wav'
y1, y2, sr = read_stereo_audio(data_fpath)
y1 = y1[-sr:]       # 48000
y2 = y2[-sr:]       # 48000

config_fpath = '../z_ref_phaselocknet/models/sound_localization/simplified_IHC3000/arch03/config.json'
with open(config_fpath, 'r') as config:
    CONFIG = json.load(config)

sr_cochlea = sr
config_filterbank = CONFIG['kwargs_cochlea']['config_filterbank']
print(config_filterbank)
filterbank_mode = config_filterbank.pop('mode', None)

### Already implemented tensorflow model

In [ ]:
# Direct one
cfs, fir, tf_1_output = fir_gammatone_filterbank(y1, sr_cochlea, **config_filterbank, return_io_function=False)

print(f'cfs : {cfs}, fir : {fir}')
print(tf_1_output.shape)
t_bin = np.linspace(0, tf_1_output.shape[-1]/sr, sr)
plot_2d(x=t_bin, y=cfs, z=to_numpy_from_tf(tf_1_output), name='tensor flow output', title=f'tensor flow cochela output',
                    xaxis_title='Time (sec)', yaxis_title='Frequency (Hz)', width=800, height=600, )

### Torch replication

In [ ]:
# torch one
# 1. erbspace
freq_min = CONFIG['kwargs_cochlea']['config_filterbank']['min_cf']
freq_max = CONFIG['kwargs_cochlea']['config_filterbank']['max_cf']
num = CONFIG['kwargs_cochlea']['config_filterbank']['num_cf']
cfs_tch = erbspace_tch(freq_min, freq_max, num)
print(f'center frequency difference btw torch and tensorflow : {cfs_tch - cfs}')

# Direct torch
fir_dur = config_filterbank['fir_dur']
firgammatonefilterbank = FIRGammatoneFilterbank(sr, fir_dur, fc=cfs_tch)
tch_1_output = firgammatonefilterbank(torch.tensor(y1))

In [ ]:
y = torch.randn(1, 1, 16000)
tf_hilbert_output = tf.math.abs(tf_hilbert(y))
print(tf_hilbert_output)
tch_hilbert_output = _hilbert_envelope(y)
print(tch_hilbert_output)
print(tf_hilbert_output - tch_hilbert_output)

In [ ]:
print(tch_1_output.shape)
# 3. get
plot_diff(t_bin, cfs, tf_1_output, tch_1_output)

In [ ]:
print(fir.shape)

fir_tch = firgammatonefilterbank.weight.squeeze().detach().numpy()
print(fir_tch.shape)

idx = 3

data_1 = fir[idx]
data_2 = np.flip(fir_tch[idx])
data_3 = data_1 - data_2
fig = px.line(data_1, title='Filter comparison: TF vs Torch')
fig.data[0].update(name='TF', line=dict(color='blue'))
fig.add_trace(px.line(data_2).data[0])
fig.data[-1].update(name ='Torch', line=dict(color='red'))
fig.show()

fig = px.line(data_3, title='Difference')
fig.show()

### 1-1. Relu

In [ ]:
tf_1_1_output = tf.nn.relu(tf_1_output)
tch_1_1_output = torch.nn.functional.relu(tch_1_output)

plot_diff(t_bin, cfs, tf_1_1_output, tch_1_1_output)

# todo : lowpass 추가

In [ ]:
lowpass

# 2. Sigmoid Rate-Level Function

### Tensorflow

In [ ]:
config_sigmoid = CONFIG['kwargs_cochlea']['kwargs_sigmoid_rate_level_function']
print(config_sigmoid)
tf_rate_output = sigmoid_rate_level_function(tf_1_1_output, **config_sigmoid)
print(tf_rate_output.shape)

### Pytorch

In [ ]:
sigmoidratelevelfunction = SigmoidRateLevelFunction(n_channels=3, **config_sigmoid)
tch_rate_output = sigmoidratelevelfunction(tch_1_1_output)
print(tch_rate_output.shape)

### Comparison

In [ ]:
ch_idx = 2

tf_data = tf_rate_output[:,:,:, ch_idx]
print(tf_data.shape)
tch_data = tch_rate_output[:,:,:, ch_idx]
print(t_bin)
print(cfs_tch)
plot_diff(t_bin, cfs_tch, tf_data, tch_data)

# 3. Binomial Spike Generation

### Tensorflow

In [ ]:
config_spike = CONFIG['kwargs_cochlea']['kwargs_spike_generator_binomial']
print(config_spike)
tf_binomial_layer = SpikeGeneratorBinomial(sr, **config_spike)
tf_final_output = tf_binomial_layer(tf_rate_output)
print(tf_final_output.shape)

### Pytorch

In [ ]:
spikegeneratorbinomial = SpikeGeneratorBinomial_Tch(sr, **config_spike)
tch_final_output = spikegeneratorbinomial(tch_rate_output)
print(tch_final_output.shape)

In [ ]:
ch_idx = 1

tf_data = tf_final_output[:,:,:, ch_idx]
print(tf_data.shape)
tch_data = tch_final_output[:,:,:, ch_idx]
plot_diff(t_bin, cfs_tch, tf_data, tch_data)

In [ ]:
ch_idx = 1

tf_data = tf_final_output[:,:,:, ch_idx]
print(tf_data.shape)
tch_data = tch_final_output[:,:,:, ch_idx]
plot_diff(t_bin, cfs_tch, tf_data, tch_data)